# predykt — runnable quick start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HishamSalem/predykt/blob/main/examples/predykt_quickstart.ipynb)

Every example from the [README](https://github.com/HishamSalem/predykt), executable end to end.

**predykt is alpha (0.x).** APIs may change between minor versions without deprecation.

### What this notebook is for

Each tool here answers a question about a *feature interaction* that a raw SHAP
value cannot:

| Section | Question |
|---|---|
| 1. `CyclicalBinner` | What is the IV-maximising split of a circular domain (hours, months)? |
| 2. `InteractionTester` | Is this interaction distinguishable from an **additive null**? |
| 3. `InteractionVoter` | Do independent algorithms agree, each against its own null? |
| 4. `SeedRobustnessValidator` | Is my metric stable across random seeds? |
| 5. `ResidualRepresentationTester` | *Which functional form* carries the interaction? |
| 6. `SHAPInteractionAnalyzer` | How do group attributions change once cross-group interactions are netted out? |

**Runtime.** `FAST = True` (the default) trims sample size and replicate counts
so the whole notebook runs in a few minutes on a free Colab CPU. Set `FAST = False`
for the README's production-scale settings — considerably slower, same API.

## Install

Works whether you opened this from GitHub or cloned the repo first.

In [ ]:
import pathlib, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/HishamSalem/predykt.git@main"

def sh(*args):
    print(">", " ".join(args[2:]))
    subprocess.run(list(args), check=True)

# If we are sitting inside a clone, install that; otherwise pull from GitHub.
local = next((c for c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (c / "pyproject.toml").exists()
              and 'name = "predykt"' in (c / "pyproject.toml").read_text()), None)

try:
    import predykt  # already present?
    print(f"predykt {predykt.__version__} already importable")
except ImportError:
    target = f"{local}[plot]" if local else f"predykt[plot] @ git+{REPO}"
    print(f"installing from {'local clone: ' + str(local) if local else 'GitHub'}")
    sh(sys.executable, "-m", "pip", "install", "-q", "-e" if local else "--upgrade", target)

# Model libraries used by the examples. Colab ships xgboost and lightgbm; catboost
# it does not. shap, numba and statsmodels come in with predykt itself.
for mod, spec in [("xgboost", "xgboost>=1.7"), ("lightgbm", "lightgbm>=3.3"),
                  ("catboost", "catboost>=1.2")]:
    try:
        __import__(mod)
    except ImportError:
        sh(sys.executable, "-m", "pip", "install", "-q", spec)

print("\ninstall step complete")

### Environment check

predykt requires `scikit-learn>=1.6`, with **no upper cap on
scikit-learn** — so nothing here should have downgraded a preinstalled package. If
this cell reports a downgrade, restart the runtime before continuing
(*Runtime → Restart session*), because the old module stays loaded in memory.

In [ ]:
import sys, importlib.metadata as md_

def v(p):
    try:    return md_.version(p)
    except Exception: return "not installed"

print(f"python        {sys.version.split()[0]}")
for p in ["predykt", "scikit-learn", "shap", "numba", "numpy",
          "pandas", "scipy", "statsmodels", "xgboost", "lightgbm", "catboost"]:
    print(f"{p:<14}{v(p)}")

from packaging.version import Version
assert Version(v("scikit-learn")) >= Version("1.6"), "scikit-learn >= 1.6 required"
print("\nenvironment OK")

## Speed switch

Set `FAST = False` to reproduce the README's production-scale numbers.

In [ ]:
FAST = True

if FAST:
    N_ROWS, N_EST, N_NULL, N_BOOT, N_SEEDS, N_PERM = 2000, 100, 30, 20, 30, 50
    VOTE_NULL, VOTE_BOOT, VOTE_EST = 50, 20, 60
else:
    N_ROWS, N_EST, N_NULL, N_BOOT, N_SEEDS, N_PERM = 4000, 200, 100, 100, 100, 100
    VOTE_NULL, VOTE_BOOT, VOTE_EST = 100, 100, 200

# alpha can never be met if it sits below the smallest attainable p-value.
ALPHA = 0.05
for label, k in [("n_null", N_NULL), ("VOTE_NULL", VOTE_NULL)]:
    assert 1 / (k + 1) < ALPHA, f"{label} must exceed {int(1/ALPHA) - 1} for alpha={ALPHA}"
print(f"FAST={FAST} | rows={N_ROWS} n_null={N_NULL} n_bootstrap={N_BOOT}")
print(f"smallest attainable p-value: section 2 = {1/(N_NULL+1):.4f}, "
      f"section 3 = {1/(VOTE_NULL+1):.4f}  (alpha = {ALPHA})")

---
## 0. The example dataset

One synthetic credit-risk frame, used by every section below. It is built so that:

- **`(age, income)` genuinely interacts** — low income hurts the young far more
  than the old. This is the signal the tools should find.
- **every other pair is additive** — honest negative controls. A tool that flags
  these is producing false positives.
- **`hour_bin` carries a 22:00–02:59 risk spike** — a window that *wraps midnight*,
  so no ordinary binner can express it as one interval.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = N_ROWS

age              = rng.integers(21, 71, n).astype(float)
income           = np.round(rng.lognormal(10.9, 0.45, n), -2)
utilization_rate = rng.beta(2, 5, n)
delinquencies    = rng.poisson(0.4, n).astype(float)
loan_amount      = np.round(income * rng.uniform(0.2, 1.5, n), -2)
tenure           = rng.integers(0, 240, n).astype(float)
hour_bin         = rng.integers(0, 24, n).astype(float)
month            = rng.integers(1, 13, n).astype(float)

z_age    = (age - age.mean()) / age.std()
z_income = (np.log(income) - np.log(income).mean()) / np.log(income).std()

night = ((hour_bin >= 22) | (hour_bin <= 2)).astype(float)

logit = (-2.6
         - 0.35 * z_age
         - 0.55 * z_income
         + 1.60 * utilization_rate
         + 0.40 * delinquencies
         + 1.00 * night                      # <- the wrap-around effect
         - 1.10 * z_age * z_income)          # <- the interaction to recover

y = pd.Series(rng.binomial(1, 1 / (1 + np.exp(-logit))), name="default")
X = pd.DataFrame({
    "age": age, "income": income, "utilization_rate": utilization_rate,
    "delinquencies": delinquencies, "loan_amount": loan_amount,
    "tenure": tenure, "hour_bin": hour_bin, "month": month,
})

# Categorical variant, used only by the adapter example in section 5.
X_cat = X.assign(
    state=rng.choice(["CA", "NY", "TX", "FL"], n),
    segment=rng.choice(["prime", "near_prime", "subprime"], n),
)

print(f"X: {X.shape}   default rate: {y.mean():.1%}")
X.head()

**Sanity check on the ground truth.** A likelihood-ratio test should find the planted interaction and *not* find the control pair.

In [ ]:
import statsmodels.api as sm
from scipy import stats

base = sm.add_constant(np.column_stack([z_age, z_income, utilization_rate, delinquencies, night]))
m0 = sm.Logit(y, base).fit(disp=0)

def lr_test(extra, label):
    m1 = sm.Logit(y, np.column_stack([base, extra])).fit(disp=0)
    stat = 2 * (m1.llf - m0.llf)
    print(f"  {label:<34} LR={stat:7.1f}   p={stats.chi2.sf(stat, 1):.3g}")
    return stats.chi2.sf(stat, 1)

p_true = lr_test(z_age * z_income,                  "(age, income)  <- planted")
p_ctrl = lr_test(utilization_rate * delinquencies,  "(utilization, delinquencies)")

assert p_true < 1e-6, "the planted interaction should be unmistakable"
assert p_ctrl > 0.01, "the control pair should look null"
print("\nground truth confirmed")

---
## 1. Cyclical Optimal Binning

Standard binners treat hour 23 and hour 0 as maximally distant. `CyclicalBinner`
treats the domain as circular and finds the IV-maximising partition accordingly.

Watch for a bin printed as `[22, 1)*` — the `*` marks the interval that wraps
past midnight. That is the partition an ordinary binner cannot represent.

In [ ]:
from predykt import CyclicalBinner

binner = CyclicalBinner(m=24, gamma=0.02, k_max=6)
binner.fit(X["hour_bin"].to_numpy(int), y)

print(f"Optimal bins: {binner.n_bins_}")
print(f"Split points: {binner.split_points_}")
print(f"IV: {binner.iv_:.4f}")
binner.result_.summary()

In [ ]:
summary = binner.result_.summary()
wrapped = [str(r) for r in summary["range"] if "*" in str(r)]
print("wrap-around bin found:", wrapped or "NONE  <- unexpected; the night effect should force one")
assert wrapped, "the planted 22:00-02:59 spike should produce a wrapping bin"

# WOE convention is ln(%non-event / %event): the high-risk bin gets a NEGATIVE woe.
rows = summary[summary["range"].astype(str).str.contains(r"\*", regex=True)]
print(f"\nits event rate {rows['event_rate'].iloc[0]:.3f} vs overall {y.mean():.3f}"
      f"  ->  woe {rows['woe'].iloc[0]:+.3f} (negative = higher risk)")

In [ ]:
binned = binner.transform(X["hour_bin"].to_numpy(int))
woe    = binner.transform_woe(X["hour_bin"].to_numpy(int))
print("bin index :", binned[:12])
print("woe values:", np.round(woe[:6], 4))

---
## 2. InteractionTester — testing against an additive null

The core idea. A large SHAP interaction value is not evidence of an interaction:
tree models manufacture interaction-shaped structure even when the target is
purely additive. `InteractionTester` builds a **simulated additive null** — refit
on a target with the same main effects and no interaction — and asks whether the
observed statistic exceeds it.

`P_Value` is the proportion of null replicates at least as extreme as observed.
Its floor is `1 / (n_null + 1)`, which is why `n_null` sets the resolution.

In [ ]:
from xgboost import XGBClassifier
from predykt import InteractionTester

tester = InteractionTester(
    model_class=XGBClassifier,
    base_params={
        "n_estimators": N_EST,
        "max_depth": 5,
        "eval_metric": "logloss",
        "verbosity": 0,
    },
    seed_param="random_state",
    n_null=N_NULL,          # additive-null replicates; drives the p-value
    n_bootstrap=N_BOOT,     # descriptive interval only; cheap to reduce
    alpha=ALPHA,
    n_jobs=4,
)

top_pairs = tester.get_top_n_interactions(X, y, n=6)   # cheap single-fit screen
print("candidate pairs:", top_pairs)

In [ ]:
results = tester.test_pairs(X, y, top_pairs)
df = tester.results_to_dataframe(results, correction_method="fdr_bh")
df[["Feature_i", "Feature_j", "Mean_Abs_Interaction", "P_Value",
    "OOF_Interaction_AUC", "Robust"]]

In [ ]:
hit = df[((df.Feature_i == "age") & (df.Feature_j == "income")) |
         ((df.Feature_i == "income") & (df.Feature_j == "age"))]
if len(hit):
    print(f"(age, income): p = {hit.P_Value.iloc[0]:.4f}   robust = {bool(hit.Robust.iloc[0])}")
else:
    print("(age, income) did not make the top-6 screen at this sample size")
print(f"\npairs flagged robust: {int(df.Robust.sum())} of {len(df)}")
print("Note: OOF_Interaction_AUC near 0.50 means the term alone does not rank-order "
      "the target,\nwhich is a narrower question than whether it improves the model.")

`plot_interaction_distribution` shows the observed statistic against the null it was tested on.

In [ ]:
import matplotlib
matplotlib.use("Agg") if False else None   # keep inline plots in the notebook
import matplotlib.pyplot as plt

tester.plot_interaction_distribution(results[0])
plt.show()

---
## 3. Cross-algorithm voting

One model's inductive bias is not evidence. `InteractionVoter` runs the same
additive-null test under several algorithms, **each against its own null**, and
reports agreement. Unanimity across independent biases is a far stronger signal
than a single p-value.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from predykt import InteractionVoter

configs = {
    "xgb":  {"model_class": XGBClassifier,
             "params": {"n_estimators": VOTE_EST, "max_depth": 4,
                        "eval_metric": "logloss", "verbosity": 0},
             "seed_param": "random_state"},
    # RandomForest is kept shallow on purpose: the cost of a SHAP interaction
    # pass grows sharply with tree depth (measured ~12x going from depth 4 to 6).
    "rf":   {"model_class": RandomForestClassifier,
             "params": {"n_estimators": VOTE_EST // 2, "max_depth": 4},
             "seed_param": "random_state"},
    "lgbm": {"model_class": LGBMClassifier,
             "params": {"n_estimators": VOTE_EST, "max_depth": 4, "verbose": -1},
             "seed_param": "random_state"},
}

voter = InteractionVoter(configs, n_bootstrap=VOTE_BOOT, n_null=VOTE_NULL,
                         alpha=ALPHA, n_jobs=4, random_state=0)
vote_results = voter.vote(X, y, [("age", "income"), ("utilization_rate", "delinquencies")])
voter.summary(vote_results)

In [ ]:
vs = voter.summary(vote_results).set_index(["Feature_i", "Feature_j"])
print(f"(age, income)                  votes {vs.loc[('age','income'),'Votes']}/3")
ctrl = vs.loc[("utilization_rate", "delinquencies"), "Votes"]
print(f"(utilization_rate, delinquencies) votes {ctrl}/3   <- negative control")
if 0 < ctrl < 3:
    print("\n  Note: an individual algorithm did flag the null pair. That is the "
          "\n  argument for voting — consensus rejected what one model's inductive "
          "\n  bias accepted.")
print(f"""
Resolution matters more than it looks. The p-value floor is 1/(n_null+1)
= {1/(VOTE_NULL+1):.4f} here. Had n_null been 20, the floor would be 0.0476 —
just under alpha=0.05 — so any pair beating all 20 replicates would score
"significant" by a hair. That knife-edge produces false positives on null
pairs. Raise n_null for resolution; n_bootstrap does not help.""")

> RandomForest is worth including deliberately: scikit-learn's forests return SHAP
> interaction values with a trailing class axis, `(n, p, p, n_classes)`, where
> XGBoost and LightGBM return `(n, p, p)`. predykt normalises both. This example
> is the regression test for that.

---
## 4. Seed robustness

Before trusting any of the above, establish how much your metric moves on random
seed alone. If seed noise exceeds the effect you are chasing, nothing downstream
is interpretable.

`sigma_max` is a **domain decision**, not a default — it is the largest metric
standard deviation you are willing to tolerate. The chi-square test asks whether
observed variance exceeds it.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from predykt import SeedRobustnessValidator

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=999)

# RandomForest, not XGBoost, and deliberately so: XGBoost with the default
# subsample=1.0 / colsample_bytree=1.0 is fully deterministic, so every seed
# returns the identical AUC and this section would report std = 0.000000.
# RandomForest bootstraps its samples, so the seed genuinely moves the metric.
hp_config = {"n_estimators": N_EST, "max_depth": 6, "min_samples_split": 10}

def eval_fn(seed: int) -> float:
    clf = RandomForestClassifier(**hp_config, random_state=seed, n_jobs=-1)
    clf.fit(X_train, y_train)
    return roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])

validator = SeedRobustnessValidator(
    eval_fn=eval_fn,
    n_seeds=N_SEEDS,
    metric_name="AUC",
    higher_is_better=True,
    sigma_max=0.005,   # domain-informed: 0.5% AUC std acceptable for production
)
report = validator.run()
validator.print_report(report)

assert report.std > 0, ("seed std of exactly 0 means the learner is deterministic "
                        "and this section is measuring nothing")

> **Read the verdict carefully.** You may see an observed std slightly *above*
> `sigma_max` while the verdict still reads ROBUST. That is not a contradiction:
> the chi-square line says *"cannot reject H0"*, not *"H0 is true"*. With 30 seeds
> the test simply lacks the power to rule out `sigma <= sigma_max`. Failure to
> reject is not proof of acceptance — the tolerance interval is the more useful
> number here, since it states the range 95% of future seed runs should fall in.


---
## 5. Residual representation testing — *which* functional form?

Section 2 answers "is there an interaction". This answers "**what shape is it**".

Stage 1 cross-fits a model on the two features and takes out-of-fold residuals.
Stage 2 regresses those residuals on each candidate representation. Whichever
survives BH correction with the largest statistic is the winning form.

`refute()` then applies two checks: a **placebo permutation** (does the signal
survive shuffling the residuals?) and **subsample stability** (does it hold on
80% subsamples?). `robust` requires all three.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from predykt import ResidualRepresentationTester, OLSEstimator

reps = pd.DataFrame({
    "product":   z_age * z_income,
    "ratio":     age / (np.log(income)),
    "log_ratio": np.log(age) - np.log(np.log(income)),
    "noise":     rng.normal(size=len(X)),      # deliberate placebo
}, index=X.index)

rr = ResidualRepresentationTester(
    model=GradientBoostingClassifier(n_estimators=N_EST, random_state=0),
    criterion=[OLSEstimator()], n_folds=5, alpha=ALPHA, random_state=0)
rr.fit(feature_pairs=[("age", "income")], X=X, y=y, representations=reps)
rr.results_to_dataframe()[["representation", "beta", "statistic", "pvalue_bh", "rejected"]]

In [ ]:
rr.refute(n_permutations=N_PERM, n_bootstrap=30)
rr.results_to_dataframe()[["representation", "pvalue_bh", "empirical_pvalue",
                           "stability_score", "rejected", "robust"]]

In [ ]:
win = rr.winning_representations()[("age", "income")]
print(f"winner        : {win['representation']}")
print(f"beta          : {win['beta']}")
print(f"pvalue_bh     : {win['pvalue_bh']}")
print(f"robust        : {win['robust']}")
print("\nWINNER'S CURSE: with K representations tested, the winning statistic is "
      "upward-biased.\nUse it for ranking, not as an effect-size estimate.")

noise_row = rr.results_to_dataframe().set_index("representation").loc["noise"]
print(f"\nplacebo column 'noise' -> rejected={noise_row['rejected']}, "
      f"robust={noise_row['robust']}  (both should be False)")

**Native-categorical models.** Wrap a CatBoost / LightGBM / XGBoost model in an adapter so fit/predict dtype handling stays consistent across folds.

In [ ]:
from catboost import CatBoostClassifier
from predykt import CatBoostAdapter

adapter = CatBoostAdapter(
    CatBoostClassifier(iterations=N_EST, depth=5, verbose=0),
    cat_cols=["state", "segment"],
)
cat_tester = ResidualRepresentationTester(model=adapter, n_folds=3, random_state=0)
cat_tester.fit(feature_pairs=[("age", "income")], X=X_cat, y=y, representations=reps)
cat_tester.results_to_dataframe()[["representation", "statistic", "pvalue_bh", "rejected"]]

---
## 6. SHAP interaction analyzer — three layers of group attribution

When features are grouped (say, by data source or business domain), the naive
group total double-counts effects shared across groups. The three layers
separate them:

- **Layer 1** — group total SHAP. Always valid.
- **Layer 2** — Layer 1 minus cross-group interactions.
- **Layer 3** — pure main effects, the diagonal of the interaction matrix.

The gap between layers *is* the cross-group interaction structure.

In [ ]:
from predykt import SHAPInteractionAnalyzer

groups = {
    "demographic":  ["age", "income"],
    "credit":       ["utilization_rate", "delinquencies", "loan_amount"],
    "temporal":     ["tenure", "hour_bin", "month"],
}

fitted_model = XGBClassifier(n_estimators=N_EST, max_depth=4,
                             eval_metric="logloss", verbosity=0).fit(X, y)

sa = SHAPInteractionAnalyzer(interaction_groups=groups, layers=[1, 2, 3])
sa.fit(model=fitted_model, X=X)

group_cmp, feat_cmp = sa.compare_layers()
group_cmp

In [ ]:
print(sa.summary(layer=1))

---
## Wrap-up

In [ ]:
print("all sections executed")
print(f"""
what the notebook demonstrated
  1. CyclicalBinner recovered a bin wrapping midnight, IV = {binner.iv_:.4f}
  2. InteractionTester tested {len(df)} pairs against a simulated additive null
  3. InteractionVoter cross-checked across XGBoost / RandomForest / LightGBM
  4. SeedRobustness measured AUC spread over {N_SEEDS} seeds: std = {report.std:.5f}
  5. ResidualRepresentationTester identified the winning functional form and
     refuted a deliberate placebo column
  6. SHAPInteractionAnalyzer separated group totals from pure main effects

settings: FAST={FAST}, rows={N_ROWS}, n_null={N_NULL}, n_bootstrap={N_BOOT}
""")

### Caveats worth carrying forward

- **`P_Value` has a floor of `1 / (n_null + 1)`.** With `n_null=30` nothing can be
  significant below 0.032. Raising `n_null` buys resolution; raising `n_bootstrap`
  does not.
- **The bootstrap interval is descriptive.** It is computed on refits of the same
  data and is biased upward relative to a true sampling distribution.
- **Winner's curse (section 5)** — the winning statistic is upward-biased when
  several representations were tested.
- **`OOF_Interaction_AUC`** answers whether the interaction term *alone*
  rank-orders the target, which is narrower than whether it improves the model.

Found a bug, or a claim that does not hold up?
[Open an issue](https://github.com/HishamSalem/predykt/issues) — that is exactly
what an alpha is for.